# Problem Set 4: Mixed effects models

*Binary inference judgments and bounded certainty ratings*


## Repeated judgments

Each participant judges several sentences, and each predicate and complement content receives judgments from several participants. A mixed effects model represents the dependence associated with these repeated units. Exercises 1–3 fit logistic models to binary judgments with `lme4`. Exercises 4–6 develop an ordered beta model of slider ratings and fit it in Stan through `brms`.

All six exercises are required. The points sum to 100. Complete the equations in **Your LaTeX**, replace every `___`, and answer the questions in **Your result**. Every numerical answer must be produced by visible code.

#### Assignment files
[Download Problem Set 4](https://aaronstevenwhite.io/stats-in-ling/downloads/ps4-assignment.zip). Extract the package, open [`ps4.ipynb`](ps4.ipynb), and select the R kernel. The [README](README.md) describes installation and runtime. The package contains [binary judgments](data/inference-choices.csv), [slider ratings](data/projection-ratings.csv), [complement contents](data/contents.csv), and [data documentation](data/README.md).

For each derivation, keep the displayed starting equations and fill the indicated steps. Show the substitution or cancellation requested at each step; you do not need to find a different proof. The numerical examples check your algebra before you apply it to the data.

Read [structured data](https://aaronstevenwhite.io/stats-in-ling/regression-models/structured-data.html), [random effects](https://aaronstevenwhite.io/stats-in-ling/regression-models/random-effects.html), and [generalized linear mixed models](https://aaronstevenwhite.io/stats-in-ling/regression-models/generalized-linear-mixed-models.html). The `ordbetareg` package defines an ordered beta response family for `brms`; it uses Stan for sampling. It does not replace the mixed model formula.


In [ ]:
library(lme4)
library(brms)
library(ordbetareg)
library(cmdstanr)
library(posterior)
library(dplyr)
library(ggplot2)
source("scripts/helpers.R")
dir.create("stan_cache", showWarnings = FALSE)
options(cmdstanr_write_stan_file_dir = normalizePath("stan_cache"))
options(mc.cores = min(4L, parallel::detectCores()))


## Exercise 1: Identify the repeated units (10 points)

The first table contains binary judgments from Experiment 1b of Degen and Tonhauser's ([2022](https://doi.org/10.1353/lan.0.0271)) *Are there factive predicates? An empirical investigation*. Participants read questions containing clause-embedding predicates and answered a question about the questioner's certainty in the complement content. The second table contains slider judgments from Experiment 2b of their ([2021](https://doi.org/10.1162/opmi_a_00042)) *Prior beliefs modulate projection*. In that experiment, a preceding fact made the complement content more or less likely. Both tables retain the authors' participant exclusions and omit control trials; they come from the [authors' repository](https://github.com/judith-tonhauser/projective-probability).

**Projection** is the inference that a speaker is committed to an embedded content even when the sentence does not assert that content. A question containing *know*, for instance, might suggest commitment to its complement despite being a question. The recorded judgments measure attributed certainty under a particular prompt; they do not directly observe a predicate's semantic representation.

### 1.1 Inspect the binary choices

The table contains all twenty predicates. To keep the fixed contrasts manageable, this analysis focuses on six: *know*, *think*, *discover*, *say*, *hear*, and *announce*. The *know–think* contrast is the target of the test below.


In [ ]:
choice_all <- read.csv("data/inference-choices.csv")
choice <- choice_all |>
  filter(predicate %in% c("know", "think", "discover", "say", "hear", "announce")) |>
  mutate(participant = factor(participant), content = factor(content),
         predicate = relevel(factor(predicate), "think"))
stopifnot(nrow(choice_all) == 8720, nrow(choice) == 2616, all(choice$yes %in% 0:1))
choice |> summarize(judgments = n(), participants = n_distinct(participant),
                    predicates = n_distinct(predicate), contents = n_distinct(content))
choice |> count(participant) |> count(n)
choice |> group_by(predicate) |> summarize(n = n(), yes_rate = mean(yes))
choice |> count(participant, content) |> count(n)


Identify the observation, participant, predicate, and content. Explain why participant and content are crossed rather than nested: a content is judged by multiple participants, and a participant judges multiple contents. Does any predicate have only yes or only no responses? Why would that matter for an unpenalized logistic coefficient?

### 1.2 Inspect the slider ratings


In [ ]:
rating <- read.csv("data/projection-ratings.csv") |>
  mutate(participant = factor(participant), content = factor(content),
         predicate = factor(predicate),
         high_c = as.integer(prior_condition == "factH") - .5)
stopifnot(nrow(rating) == 5320, all(rating$rating >= 0 & rating$rating <= 1))
rating |> summarize(judgments = n(), participants = n_distinct(participant),
                    predicates = n_distinct(predicate), contents = n_distinct(content))
rating |> count(prior_condition)
rating |> group_by(participant) |>
  summarize(conditions = n_distinct(high_c), .groups = "drop") |> count(conditions)
rating |> group_by(predicate, content) |>
  summarize(conditions = n_distinct(high_c), .groups = "drop") |> count(conditions)


`high_c` is −0.5 for the lower probability fact and +0.5 for the higher probability fact. It codes the experimental condition, not an individual participant's measured prior belief. Identify which grouping factors contain both conditions and can support a condition slope. A fact belongs to a particular content and condition; explain why forty fact labels would not constitute forty independent replications of the condition manipulation.

The two experiments use related materials and ask about speaker certainty, but differ in preceding context and response format. Do not compare their raw means as if they were two conditions of one experiment. Participant labels have been replaced separately in each table and cannot establish whether someone took both studies.

#### Your result
Replace this sentence with your answers to Exercise 1.

## Exercise 2: Fit a mixed logistic model and derive a test (20 points)

The binary model has fixed contrasts for the six selected predicates and twenty contents, with a random intercept for participant. Let $s[i]$, $c[i]$, and $v[i]$ identify those units for judgment $i$:

$$
\begin{aligned}
Y_i\mid a_{s[i]}&\sim\operatorname{Bernoulli}(p_i),\\
\operatorname{logit}(p_i)&=\alpha+\beta_{v[i]}+\gamma_{c[i]}+a_{s[i]},\\
a_s&\sim\mathcal N(0,\tau_s^2).
\end{aligned}
$$

Write $F(z)=1/(1+e^{-z})$ for the inverse logit. Set $\beta_{\textit{think}}=0$ and one content coefficient to zero as reference. Effects for distinct participants are independent. Judgments are independent conditional on these effects and the model parameters. Marginally, judgments sharing a unit can remain dependent.

### 2.1 Derive a conditional contrast

Keep the participant effect $a$ and content effect $\gamma_c$ the same in both predictions. Here a conditional comparison means that these effects are held fixed. Complete the following steps in **Your LaTeX**.

1. Write the two log odds, using the reference constraint $\beta_{\textit{think}}=0$:

   $$
   \begin{aligned}
   \operatorname{logit}(p_{\textit{know}})&=\alpha+\beta_{\textit{know}}+\gamma_c+a,\\
   \operatorname{logit}(p_{\textit{think}})&=\alpha+0+\gamma_c+a.
   \end{aligned}
   $$

2. Subtract the second line from the first. Show which terms cancel:

   $$
   (\alpha+\beta_{\textit{know}}+\gamma_c+a)
   -(\alpha+\gamma_c+a)=\text{enter know contrast}.
   $$

3. Use $\operatorname{logit}(p)=\log[p/(1-p)]$ and $\log u-\log v=\log(u/v)$. Thus the difference in step 2 is the log of the odds ratio. Apply $\exp$ to both sides:

   $$
   \operatorname{OR}_{\textit{know}/\textit{think}}=\text{enter know odds ratio}.
   $$

As an algebra check, a coefficient of $\log 2$ must give an odds ratio of 2. It does not double the response probability: starting from probability $1/2$, doubling the odds gives probability $2/3$.

Explain why this is not a difference in response probabilities. The model treats these predicates and contents as fixed contrasts because the prediction target concerns the recorded materials. Explain why adding a random intercept for predicate alongside a separate fixed coefficient for every predicate would not identify an additional source of predicate variation.

### 2.2 Fit with lme4

The [glmer documentation](https://lme4.github.io/lme4/reference/glmer.html) describes maximum likelihood estimation by integrating over random effects. With one scalar intercept per participant, `nAGQ = 9` uses nine adaptive quadrature points to approximate each Gaussian integral. Quadrature approximates an integral with a weighted sum of function values. The optimizer estimates the fixed coefficients and participant variance jointly.


In [ ]:
fit_choice <- function(formula, data) {
  glmer(formula, data = data, family = binomial(link = "logit"),
        nAGQ = 9,
        control = glmerControl(optimizer = "bobyqa", tolPwrss = 1e-9,
                               optCtrl = list(maxfun = 200000)))
}
choice_fit <- fit_choice(yes ~ predicate + content + (1 | participant), choice)
summary(choice_fit)
lme4::VarCorr(choice_fit)
isSingular(choice_fit)
choice_fit@optinfo$conv
stopifnot(is.null(choice_fit@optinfo$conv$lme4$messages))
know_interval <- confint(choice_fit, parm = "predicateknow", method = "Wald")
c(estimate = lme4::fixef(choice_fit)["predicateknow"], exp(know_interval))


Report the coefficient, odds ratio, approximate Wald confidence interval for the odds ratio, and participant intercept SD. This [Wald interval](https://aaronstevenwhite.io/stats-in-ling/statistical-inference/confidence-intervals.html) uses the coefficient estimate plus or minus 1.96 estimated standard errors, then exponentiates both limits. A confidence interval describes a procedure's repeated sampling behavior under its assumptions; it is not a posterior interval. Successful optimization does not verify the dependence assumptions or representative sampling.

### 2.3 Derive a likelihood ratio test

Test $H_0:\beta_{\textit{know}}=0$. Under this null, *know* and *think* share a fixed coefficient; every other predicate keeps its own coefficient. Combine just those two labels to impose this restriction. Retain all content contrasts and the participant random intercept, and reestimate its variance in each fit.

Let $\widehat L_0,\widehat L_1$ be the maximized likelihoods under the null and alternative, and write $\widehat\ell_j=\log\widehat L_j$. Both likelihoods integrate over participant effects; they do not substitute the fitted participant modes into a Bernoulli likelihood. Here `glmer` approximates those integrals by quadrature.

Follow the [likelihood ratio test notes](https://aaronstevenwhite.io/stats-in-ling/model-criticism-and-prediction/likelihood-ratio-tests.html). Complete steps 2–4 in **Your LaTeX**.

1. Form the ratio $\Lambda=\widehat L_0/\widehat L_1$. The null parameter space is contained in the alternative parameter space, so $0<\Lambda\leq1$.
2. Take logs. Use $\log(u/v)=\log u-\log v$:

   $$
   \log\Lambda=\text{enter log likelihood ratio}.
   $$

3. Multiply by $-2$ and reverse the subtraction so the larger log likelihood comes first:

   $$
   D=-2\log\Lambda=\text{enter likelihood statistic}.
   $$

   Check the sign: if the two fits have log likelihoods $-105$ and $-100$, respectively, $D$ should be 10, not $-10$.

4. Count the free parameters that differ. The alternative has five predicate contrasts for six labels; the null has four for five labels. Both have an intercept, 19 content contrasts, and one participant variance. Thus the difference in dimension is $(1+5+19+1)-(1+4+19+1)$. Complete

   $$
   D\overset{H_0}{\approx}\chi^2_{\text{enter degrees of freedom}}.
   $$

5. The p value is the probability of a value at least as large as the observed $D$ under this reference distribution. Translate that upper tail into `pchisq(..., lower.tail = FALSE)` below. The null fixes a coefficient at an interior value, not a variance at its boundary; a test of zero participant variance would require a different calibration.


In [ ]:
choice$predicate_null <- factor(ifelse(choice$predicate %in% c("know", "think"),
                                      "know_or_think", as.character(choice$predicate)))
choice_null <- fit_choice(yes ~ predicate_null + content + (1 | participant), choice)
stopifnot(is.null(choice_null@optinfo$conv$lme4$messages))
likelihood_statistic <- ___
likelihood_p <- pchisq(likelihood_statistic, df = ___, lower.tail = FALSE)
c(statistic = likelihood_statistic, p_value = likelihood_p)
anova(choice_null, choice_fit)


Report the statistic and p value. Explain why the alternative's maximum cannot be smaller when it contains the null. The chi-squared calibration is asymptotic and also depends on the likelihood approximation. This comparison tests the specified predicate contrast; it does not establish a discrete semantic classification of the twenty predicates.

#### Your result
Replace this sentence with your answers to Exercise 2.

## Exercise 3: Derive partial pooling and predict for new participants (15 points)

### 3.1 Derive the equation for a participant effect

Hold all fixed coefficients and $\tau_s$ fixed. Write $\eta_i^{(-s)}=\alpha+\beta_{v[i]}+\gamma_{c[i]}$. The conditional log density for participant $s$ is, up to a constant,

$$
\ell_s(a)=\sum_{i:s[i]=s}\left[y_i(\eta_i^{(-s)}+a)
-\log(1+e^{\eta_i^{(-s)}+a})\right]-\frac{a^2}{2\tau_s^2}.
$$

For this derivative, $a$ is the only variable. The observations $y_i$, predictors $\eta_i^{(-s)}$, and positive variance $\tau_s^2$ are constants. Let $z_i=\eta_i^{(-s)}+a$, so $dz_i/da=1$. Use the [logistic score notes](https://aaronstevenwhite.io/stats-in-ling/regression-models/logistic-regression.html) and complete these steps in **Your LaTeX**.

1. Differentiate one linear term: $d(y_i z_i)/da=y_i$.
2. Differentiate one log term by the chain rule. The derivative of $\log u$ is $u'/u$, and the derivative of $e^{z_i}$ is $e^{z_i}$:

   $$
   \frac{d}{da}\log(1+e^{z_i})
   =\frac{e^{z_i}}{1+e^{z_i}}=\text{enter inverse logit derivative term}.
   $$

3. Differentiate the Gaussian term, keeping its minus sign:

   $$
   \frac{d}{da}\left(-\frac{a^2}{2\tau_s^2}\right)
   =\text{enter Gaussian penalty derivative}.
   $$

4. Add these derivatives over the participant's judgments. Set the sum to zero to obtain the equation solved by `uniroot()`:

   $$
   0=\ell_s'(a)=\text{enter conditional score}.
   $$

5. Check that a root gives a maximum. Use $F'(z)=F(z)[1-F(z)]$ and write $p_i(a)=F(\eta_i^{(-s)}+a)$:

   $$
   \ell_s''(a)=-\sum_{i:s[i]=s}p_i(a)[1-p_i(a)]
   -\text{enter penalty curvature}.
   $$

   Explain why this derivative is strictly negative for finite positive $\tau_s^2$. For all-yes responses, the likelihood part of the score is nonnegative and bounded by the participant's number of judgments. Compare it with $-a/\tau_s^2$ as $a\to\infty$ and $a\to-\infty$ to show that the total score crosses zero.

The Gaussian penalty contributes a term that pulls the conditional mode toward zero. Unlike the Gaussian calculation in the notes on [partial pooling](https://aaronstevenwhite.io/stats-in-ling/regression-models/fixed-and-random-effects.html), this equation generally requires a numerical solution. Explain why it still gives a finite mode when a participant gives only yes responses, provided $\tau_s>0$ is finite.


In [ ]:
eta_without_participant <- predict(choice_fit, re.form = NA)
tau_s <- attr(lme4::VarCorr(choice_fit)$participant, "stddev")[1]
participant_modes <- lme4::ranef(choice_fit)$participant
pooling <- lapply(levels(choice$participant), function(s) {
  rows <- which(choice$participant == s)
  score <- function(a) {
    sum(choice$yes[rows] - plogis(eta_without_participant[rows] + a)) - ___
  }
  data.frame(participant = s, n = length(rows), yes_rate = mean(choice$yes[rows]),
             derived_mode = uniroot(score, c(-30, 30), tol = 1e-10)$root,
             fitted_mode = participant_modes[s, "(Intercept)"])
}) |> bind_rows()
max(abs(pooling$derived_mode - pooling$fitted_mode))
pooling |> filter(yes_rate %in% c(0, 1))


[ranef()](https://lme4.github.io/lme4/reference/ranef.html) returns conditional modes. They need not equal conditional means in a logistic model. Compare the derived and fitted modes, allowing numerical approximation, and explain why this calculation omits uncertainty from estimating the other parameters.

### 3.2 Hold out complete participants

Read [grouped cross-validation](https://aaronstevenwhite.io/stats-in-ling/model-criticism-and-prediction/grouped-cross-validation.html) and [prediction targets](https://aaronstevenwhite.io/stats-in-ling/model-criticism-and-prediction/prediction-targets.html). Compare the mixed model with a logistic regression using the same fixed predicate and content contrasts. Each fold excludes all judgments from its test participants. The target is a new participant judging the recorded predicates and contents.

For a participant absent from training, there is no fitted individual offset to insert. Construct the marginal prediction in three steps.

1. Conditional on an intercept $a$, the yes probability is $F(\mathbf x^\top\boldsymbol\beta+a)$. The vector $\mathbf x$ contains the fixed predicate and content contrasts.
2. Average that conditional probability over $a\sim\mathcal N(0,\tau_s^2)$:

   $$
   p_{\mathrm{new}}(\mathbf x)=
   \int F(\mathbf x^\top\boldsymbol\beta+a)\,
   \varphi(a;0,\tau_s^2)\,da.
   $$

   Here $\varphi$ is the normal density, not the inverse logit $F$.
3. Write $a=\tau_s z$, where $z$ has a standard normal distribution. Approximate the average by evaluating $F$ at 2,000 equally spaced normal quantiles:

   $$
   p_{\mathrm{new}}(\mathbf x)\approx\frac1{2000}
   \sum_{g=1}^{2000}F(\mathbf x^\top\boldsymbol\beta+\tau_s z_g).
   $$

Match the fixed predictor, SD, and $z_g$ to `e`, `tau`, and `normal_grid` in the code. Averaging the probabilities is generally different from applying $F$ to the average intercept, zero, because $F$ is nonlinear. The fitted coefficients and SD are treated as estimates here; this calculation integrates a new participant effect, not uncertainty in all model parameters.


In [ ]:
set.seed(485)
participant_ids <- levels(choice$participant)
folds <- data.frame(participant = participant_ids,
                    fold = sample(rep(1:5, length.out = length(participant_ids))))
normal_grid <- qnorm(((1:2000) - .5) / 2000)
scored <- lapply(1:5, function(k) {
  ids <- folds$participant[folds$fold == k]
  train <- droplevels(choice[!choice$participant %in% ids, ])
  test <- choice[choice$participant %in% ids, ]
  mixed <- fit_choice(formula(choice_fit), train)
  pooled <- glm(yes ~ predicate + content, data = train, family = binomial())
  stopifnot(is.null(mixed@optinfo$conv$lme4$messages), pooled$converged,
            all(is.finite(coef(pooled))))
  eta <- predict(mixed, newdata = test, re.form = NA, allow.new.levels = TRUE)
  tau <- attr(lme4::VarCorr(mixed)$participant, "stddev")[1]
  mixed_p <- vapply(eta, function(e) mean(plogis(e + tau * normal_grid)), numeric(1))
  pooled_p <- predict(pooled, newdata = test, type = "response")
  loss <- function(p) -dbinom(test$yes, 1, p, log = TRUE)
  data.frame(participant = test$participant, fold = k,
             mixed_loss = loss(mixed_p), pooled_loss = loss(pooled_p),
             singular = isSingular(mixed))
}) |> bind_rows()
participant_loss <- scored |> group_by(participant) |>
  summarize(mixed = mean(mixed_loss), pooled = mean(pooled_loss),
            improvement = ___, .groups = "drop")
participant_loss |> summarize(mean_improvement = mean(improvement),
                              improved = sum(improvement > 0), participants = n())
scored |> distinct(fold, singular)


Positive improvement favors the mixed model. Report its average, how many participants improve, and any singular fold fits. A singular fit estimates a random effect dimension on a boundary; retain and report it. Does the result support a predictive benefit, or is the difference small or inconsistent? Explain why estimating variation among observed participants does not identify a new participant's offset. Do not infer a zero population variance from a small predictive difference. The training folds overlap, so the five scores are not independent observations for a t test.

#### Your result
Replace this sentence with your answers to Exercise 3.

## Exercise 4: Explain why an ordinary beta likelihood is inadequate (20 points)

Read [why ordinary beta regression fails for endpoint responses](https://aaronstevenwhite.io/stats-in-ling/regression-models/why-not-beta-sliders.html) and [ordered beta regression](https://aaronstevenwhite.io/stats-in-ling/regression-models/ordered-beta-regression.html). Grove and White's (2026) discussion of beta likelihoods ([Supplementary Materials, Section 1](https://static-content.springer.com/esm/art%3A10.1007%2Fs11050-025-09244-9/MediaObjects/11050_2025_9244_MOESM1_ESM.pdf), accompanying their [factivity paper](https://doi.org/10.1007/s11050-025-09244-9)) motivates this exercise.

An ordinary beta distribution with positive, finite shape parameters is continuous on the interior. It assigns zero probability mass to either endpoint, even when its density grows without bound near an endpoint. Density height and point probability are different quantities. The slider records exact zeros and ones, and those responses may matter for hypotheses about categorical certainty; replacing them with interior values changes the recorded judgments.

Interior observations contribute density to the likelihood; recorded endpoint events need probability masses or an explicit rounding or censoring model. If rounding produced the endpoints, integrating a beta density over rounding intervals would be a different measurement model. Evaluating a density after an arbitrary nudge does not perform that integration.

### 4.1 Count the endpoints and derive nudge sensitivity


In [ ]:
endpoint_counts <- rating |>
  summarize(zero = sum(rating == 0), interior = sum(rating > 0 & rating < 1),
            one = sum(rating == 1))
endpoint_counts


For constant shape parameters $A,B>0$, the beta log likelihood for $n$ interior values is

$$
\ell(A,B)=-n\log\mathrm B(A,B)
+(A-1)\sum_i\log y_i+(B-1)\sum_i\log(1-y_i).
$$

Here $\mathrm B$ is the beta function, which normalizes the density. A common workaround replaces each zero by $\delta$ and each one by $1-\delta$, where $0<\delta<1/2$. Let $n_0$ and $n_1$ be their observed counts. Derive the effect of this replacement in **Your LaTeX**.

1. Start from a single row's log density:

   $$
   g(y)=-\log\mathrm B(A,B)+(A-1)\log y+(B-1)\log(1-y).
   $$

2. Substitute $y=\delta$ for an original zero. Substitute $y=1-\delta$ for an original one. In the second substitution, simplify $1-y=1-(1-\delta)=\delta$:

   $$
   \begin{aligned}
   g(\delta)&=-\log\mathrm B(A,B)
   +(A-1)\log\delta+(B-1)\log(1-\delta),\\
   g(1-\delta)&=-\log\mathrm B(A,B)
   +(A-1)\log(1-\delta)+(B-1)\log\delta.
   \end{aligned}
   $$

3. Multiply the first line by $n_0$ and the second by $n_1$. Add them and collect the terms multiplying each log:

   $$
   \begin{aligned}
   n_0g(\delta)+n_1g(1-\delta)
   &=-(n_0+n_1)\log\mathrm B(A,B)\\
   &\quad+K\log\delta+J\log(1-\delta),
   \end{aligned}
   $$

   $$
   K=\text{enter log delta coefficient},\qquad
   J=\text{enter log one minus delta coefficient}.
   $$

4. Check your coefficients in two cases. For $A=B=2$, both should equal $n_0+n_1$. For $A=B=1/2$, both should equal $-(n_0+n_1)/2$. Which choice makes the endpoint contribution increase as $\delta$ approaches zero?

Since $\log\delta\to-\infty$ as $\delta\to0$, the choice of nudge can dominate the endpoint contribution. Its direction depends on the shape parameters; it need not always lower the log likelihood. Grove and White's concern is that an arbitrary response transformation can change likelihood-based model comparisons, including WAIC. It is not merely a numerical error that a smaller nudge repairs.


In [ ]:
nudge_check <- lapply(c(.01, .001, .0001, .00001), function(delta) {
  y <- rating$rating
  y[y == 0] <- delta
  y[y == 1] <- 1 - delta
  # Two fixed beta densities illustrate sensitivity; neither is a fitted regression.
  data.frame(delta = delta,
             log_density_2_2 = sum(dbeta(y, 2, 2, log = TRUE)),
             log_density_half_half = sum(dbeta(y, .5, .5, log = TRUE)))
}) |> bind_rows()
nudge_check |> mutate(difference = log_density_2_2 - log_density_half_half)


Report how the difference changes with $\delta$. Explain why comparing scores computed on different transformed responses does not evaluate prediction of the same recorded outcome. In regression, $A_i,B_i$ can vary across rows, so the corresponding sensitivity terms also vary with the predictors. This exercise uses constant shapes only to isolate the algebra.

### 4.2 Derive the ordered beta mean

Let $F$ be the inverse logit, $\eta_i$ a predictor location, $\mu_i=F(\eta_i)$, $\phi>0$ an interior precision, and $\kappa_0<\kappa_1$ two cutpoints. Define

$$
\begin{aligned}
p_{0i}&=F(\kappa_0-\eta_i),\\
p_{1i}&=1-F(\kappa_1-\eta_i),\\
p_{mi}&=F(\kappa_1-\eta_i)-F(\kappa_0-\eta_i).
\end{aligned}
$$

A zero contributes mass $p_{0i}$, a one contributes mass $p_{1i}$, and an interior observation contributes density

$$
p_{mi}\operatorname{Beta}(y_i\mid\mu_i\phi,(1-\mu_i)\phi).
$$

Use the [law of total expectation](https://aaronstevenwhite.io/stats-in-ling/random-variables-and-distributions/conditional-expectation.html) to average over the three regions. Complete these steps in **Your LaTeX**.

1. Abbreviate $F(\kappa_0-\eta_i)$ by $F_0$ and $F(\kappa_1-\eta_i)$ by $F_1$. Add the masses:

   $$
   p_{0i}+p_{mi}+p_{1i}=F_0+(F_1-F_0)+(1-F_1)=1.
   $$

2. List the conditional means. In the zero region, $Y_i$ is exactly 0; in the one region, it is exactly 1. In the interior region, the beta mean is

   $$
   \frac{\mu_i\phi}{\mu_i\phi+(1-\mu_i)\phi}=\mu_i.
   $$

3. Multiply each conditional mean by the probability of its region. Simplify:

   $$
   \begin{aligned}
   \mathbb{E}[Y_i\mid\eta_i,\phi,\boldsymbol\kappa]
   &=p_{0i}\,0+p_{mi}\,\mu_i+p_{1i}\,1\\
   &=\text{enter ordered beta mean}.
   \end{aligned}
   $$

4. Check with $p_{0i}=.2$, $p_{mi}=.5$, $p_{1i}=.3$, and $\mu_i=.4$. The full mean should be .5. State why the precision cancels from this conditional mean even though it affects the interior response distribution.

Explain why $\mu_i$ is the interior mean, rather than generally the mean of the full slider response. Ordered beta regression preserves endpoints but imposes a shared location: the same $\eta_i$ controls endpoint probabilities and the interior beta mean. It does not establish that endpoint selections reflect discrete semantic knowledge, and endpoint masses alone do not decide between discrete and gradient theories of projection.

#### Your result
Replace this sentence with your answers to Exercise 4.

## Exercise 5: Fit a Bayesian ordered beta mixed model (20 points)

### 5.1 Specify crossed intercepts and a varying slope

For the slider data, use

$$
\eta_i=\alpha+\beta_H H_i+a_{s[i]}+b_{0v[i]}+b_{1v[i]}H_i+d_{c[i]},
$$
$$
a_s\sim\mathcal N(0,\tau_s^2),\qquad d_c\sim\mathcal N(0,\tau_c^2),
$$
$$
\begin{pmatrix}b_{0v}\\b_{1v}\end{pmatrix}
\sim\mathcal N\left[
\begin{pmatrix}0\\0\end{pmatrix},
\begin{pmatrix}\tau_0^2&\rho\tau_0\tau_1\\
\rho\tau_0\tau_1&\tau_1^2\end{pmatrix}\right].
$$

$H_i$ is the centered condition indicator. Participants, predicates, and contents are crossed. The condition slope varies across predicates. Unlike Exercise 2, predicates are modeled as deviations from a shared distribution, allowing partial pooling of their condition contrasts. Participant and content condition slopes are also supported by the design but omitted here; sensitivity to the condition may thus remain incompletely modeled.

Read [random slopes](https://aaronstevenwhite.io/stats-in-ling/regression-models/random-slopes.html) and [random effect correlations](https://aaronstevenwhite.io/stats-in-ling/regression-models/random-effect-correlations.html). Isolate the covariance contributed by a shared predicate to two **location predictors**. Complete steps 2–4 in **Your LaTeX**.

1. Keep only the predicate terms: $B_i=b_{0v}+H_i b_{1v}$ and $B_j=b_{0v}+H_j b_{1v}$. Treat $H_i,H_j$ as fixed numbers. Both effects have mean zero, so $\operatorname{Cov}(B_i,B_j)=\mathbb{E}[B_iB_j]$.
2. Multiply the two sums before taking expectations:

   $$
   B_iB_j=b_{0v}^2+(H_i+H_j)b_{0v}b_{1v}
   +\text{enter slope product}.
   $$

3. Read the three second moments from the covariance matrix: $\mathbb{E}[b_{0v}^2]=\tau_0^2$, $\mathbb{E}[b_{1v}^2]=\tau_1^2$, and $\mathbb{E}[b_{0v}b_{1v}]=\rho\tau_0\tau_1$. Substitute them into step 2:

   $$
   \operatorname{Cov}_v(\eta_i,\eta_j)=\text{enter predicate covariance}.
   $$

4. For two judgments in the low condition, substitute $H_i=H_j=-1/2$. For one low and one high judgment, substitute $H_i=-1/2,H_j=1/2$. Complete

   $$
   \begin{aligned}
   \operatorname{Cov}_v(\eta_L,\eta_L')&=\text{enter low condition covariance},\\
   \operatorname{Cov}_v(\eta_L,\eta_H)&=\text{enter cross condition covariance}.
   \end{aligned}
   $$

Explain why judgments sharing only a predicate can have different covariances depending on condition. Independent effects from different predicates contribute zero covariance; shared participant or content effects would contribute additional terms. This calculation concerns the location predictors. The response link and ordered beta distribution intervene between those predictors and the observed slider responses.

### 5.2 Fit through brms

[`ordbetareg()`](https://search.r-project.org/CRAN/refmans/ordbetareg/html/ordbetareg.html) calls `brms` with a custom ordered beta family. `center = FALSE` keeps the fixed intercept at $H=0$, the midpoint of the conditions. The priors are $\alpha\sim\mathcal N(0,1.5^2)$, $\beta_H\sim\mathcal N(0,1)$, independent exponential priors with rate 1 on group SDs, an LKJ(2) prior on the predicate correlation matrix, and $\phi\sim\operatorname{Exponential}(.1)$. The cutpoint prior induces a Dirichlet(1,1,1) distribution on the three region probabilities at $\eta=0$. The Dirichlet prior is uniform over triples of nonnegative probabilities summing to one; it does not make the three probabilities independently uniform. With two predicate effects, LKJ(2) gives greater prior density near zero correlation than near ±1. These are response model priors, not measured prior beliefs about the complement.


In [ ]:
projection_fit <- ordbetareg::ordbetareg(
  formula = bf(rating ~ high_c + (1 | participant) +
                 (1 + high_c | predicate) + (1 | content), center = FALSE),
  data = rating, true_bounds = c(0, 1),
  coef_prior_SD = 1, intercept_prior_mean = 0, intercept_prior_SD = 1.5,
  phi_prior = .1, dirichlet_prior = c(1, 1, 1),
  extra_prior = c(prior(exponential(1), class = sd), prior(lkj(2), class = cor)),
  backend = "cmdstanr", chains = 4, cores = 4,
  iter = 4500, warmup = 1500, seed = 482, refresh = 0,
  control = list(adapt_delta = .99, max_treedepth = 12),
  file = "stan_cache/projection-ordered-beta", file_refit = "on_change"
)
check_fit <- function(fit) {
  draws <- summarise_draws(as_draws_array(fit))
  sampler <- nuts_params(fit)
  diagnostic <- data.frame(
    max_rhat = max(draws$rhat, na.rm = TRUE),
    min_bulk_ess = min(draws$ess_bulk, na.rm = TRUE),
    min_tail_ess = min(draws$ess_tail, na.rm = TRUE),
    divergences = sum(sampler$Value[sampler$Parameter == "divergent__"]),
    depth_hits = sum(sampler$Value[sampler$Parameter == "treedepth__"] >= 12))
  print(diagnostic)
  stopifnot(diagnostic$max_rhat < 1.01, diagnostic$min_bulk_ess > 400,
            diagnostic$min_tail_ess > 400, diagnostic$divergences == 0,
            diagnostic$depth_hits == 0)
  invisible(diagnostic)
}
check_fit(projection_fit)
brms::fixef(projection_fit)
brms::VarCorr(projection_fit)
projection_draws <- as_draws_df(projection_fit)
quantile(projection_draws$b_high_c, c(.025, .5, .975))


If a diagnostic fails, inspect the chains before interpreting the fit. To extend sampling from an existing cache, increase `iter` and set `file_refit = "always"` for that run; `"on_change"` does not detect changes to sampling iterations.

Use the [chain diagnostics notes](https://aaronstevenwhite.io/stats-in-ling/statistical-inference/checking-markov-chains.html) to interpret R-hat and bulk and tail effective sample size. Report the diagnostics, condition coefficient and interval, predicate slope SD and interval, and intercept–slope correlation and interval. A positive condition coefficient increases the probability of one, decreases the probability of zero, and increases the interior mean at fixed group effects. It is not itself a change in the mean slider rating. Explain why a small slope SD can make its intercept–slope correlation difficult to estimate. Do not assign a reliable direction to an interval spanning zero, or use a positive SD interval as a test of an exactly zero variance component.

#### Your result
Replace this sentence with your answers to Exercise 5.

## Exercise 6: Check endpoint use and conditional response distributions (15 points)

Read [posterior predictive checks](https://aaronstevenwhite.io/stats-in-ling/statistical-inference/posterior-predictive-checks.html). A model can match the overall mean while missing endpoint use or differences among predicates and conditions. Check both the mean and endpoint rates within each predicate–condition combination.


In [ ]:
replicated <- ordered_beta_replications(projection_fit, ndraws = 400, seed = 486)
cell <- interaction(rating$predicate, rating$prior_condition, drop = TRUE)
checks <- lapply(levels(cell), function(g) {
  rows <- which(cell == g)
  observed <- rating$rating[rows]
  rep <- replicated$rating[, rows, drop = FALSE]
  statistics <- list(mean = rowMeans(rep),
                     zero = rowMeans(replicated$zero[, rows, drop = FALSE]),
                     one = rowMeans(replicated$one[, rows, drop = FALSE]))
  actual <- c(mean = mean(observed), zero = mean(observed == 0), one = mean(observed == 1))
  bind_rows(lapply(names(statistics), function(name) {
    limits <- quantile(statistics[[name]], c(.025, .5, .975))
    data.frame(cell = g, statistic = name, n = length(rows), observed = actual[name],
               lower = limits[1], median = limits[2], upper = limits[3])
  }))
}) |> bind_rows()
ggplot(checks, aes(observed, median)) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed") +
  geom_linerange(aes(ymin = lower, ymax = upper)) + geom_point() +
  facet_wrap(~ statistic, scales = "free") +
  labs(x = "Observed statistic", y = "Replicated statistic")
checks |> mutate(outside = observed < lower | observed > upper) |>
  group_by(statistic) |> summarize(outside = sum(outside), cells = n())


The helper in [`scripts/helpers.R`](scripts/helpers.R) records which region generated each response. This keeps floating point rounding of an interior beta draw from being counted as endpoint mass; it changes no observed response.

Identify the two largest discrepancies, describe their direction, and assess whether they are substantial relative to the replicated ranges. These are pointwise ranges across many related checks, not a simultaneous acceptance test. A discrepancy may reflect an inadequate mean structure, missing dependence, or the shared location restriction; it does not uniquely diagnose a cause.

Finally, compare the full expected response with the interior mean. The package calls its location parameter `mu`, but its identity link makes that parameter $\eta$ in our notation. Its `posterior_epred()` method returns the full ordered beta expectation derived in Exercise 4.


In [ ]:
expected <- posterior_epred(projection_fit, draw_ids = replicated$draw_ids)
location <- posterior_linpred(projection_fit, draw_ids = replicated$draw_ids, transform = FALSE)
mean_difference <- colMeans(expected) - colMeans(plogis(location))
range(mean_difference)


Report the range. Explain why using only the inverse logit of the location would give the wrong squared error prediction in general. The posterior predictive checks condition on the observed participants, predicates, and contents; they are not a test of prediction for new participants. Contrast that target with the complete-participant folds in Exercise 3, and describe how a new predicate would require integrating a new intercept and condition slope together.

#### Your result
Replace this sentence with your answers to Exercise 6.

Submit the completed notebook with all six result blocks, equations, and visible code output. Confirm that its cells run in order from a fresh R session; keep fitted model files out of the submission.
